# Hidden Unit Sweep with Multi-Seed Variance — v8

Sweeps the hidden-unit count from 1 to 8 for the **single 4-class unified network**, with 5
random seeds per size. Produces:

1. the **variance graph**: overall test accuracy (mean +/- std over seeds) against hidden units, which
   is what the hidden-unit count is chosen from;
2. a **per-subtask** version: final test accuracy against hidden units for each of the 12 subtasks,
   in two colour groups (detection vs localisation).

In [ ]:
import os
os.environ["MKL_DISABLE_FAST_MM"] = "1"   # optional
import warnings
warnings.filterwarnings("ignore")

## 1. Setup (MPS-enabled)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
import time

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda"); print("Using GPU (CUDA):", torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    device = torch.device("mps"); print("Using GPU (MPS - Apple Silicon)")
else:
    device = torch.device("cpu"); print("WARNING: no GPU acceleration, using CPU (slower).")

DATA_DIR = Path("./generated_trials_v8")

MODEL_DIR = Path("./trained_models_v8_sweep"); MODEL_DIR.mkdir(exist_ok=True)

In [ ]:
import torch
device = torch.device("cpu")
torch.set_num_threads(torch.get_num_threads())   # use all cores
print("Forcing device:", device)

## 2. Load data and define subtask groups

In [ ]:
DET_SUBTASKS = ["det_absent","det_auditory_only","det_visual_only","det_multisensory"]
LOC_SUBTASKS = ["loc_auditory_only_L","loc_auditory_only_R","loc_visual_only_L","loc_visual_only_R",
                "loc_multisensory_same_L","loc_multisensory_same_R",
                "loc_conflict_audL_visR","loc_conflict_audR_visL"]
SUBTASKS = DET_SUBTASKS + LOC_SUBTASKS
TEST_ONLY = {"det_multisensory"}            # not trained
CONFLICT  = {"det_multisensory","loc_conflict_audL_visR","loc_conflict_audR_visL"}

# Colour each subtask by group: detection = oranges, localisation = blues.
_det_cols = plt.cm.Oranges(np.linspace(0.45, 0.92, len(DET_SUBTASKS)))
_loc_cols = plt.cm.Blues(np.linspace(0.35, 0.95, len(LOC_SUBTASKS)))
SUBTASK_STYLE = {}
for i, s in enumerate(DET_SUBTASKS):
    SUBTASK_STYLE[s] = dict(color=_det_cols[i], ls=("--" if s in CONFLICT else "-"))
for i, s in enumerate(LOC_SUBTASKS):
    SUBTASK_STYLE[s] = dict(color=_loc_cols[i], ls=("--" if s in CONFLICT else "-"))

def load_dataset(filename):
    d = np.load(DATA_DIR / filename, allow_pickle=True)
    return {"X": d["X"].astype(np.float32), "y": d["y_4class"].astype(np.int64),
            "types": d["types"], "n_classes": int(d["n_classes"])}

train = load_dataset("train.npz")
test  = load_dataset("test.npz")
X_tr, y_tr = train["X"], train["y"]
X_te, y_te, types_te = test["X"], test["y"], test["types"]
TEST_MASKS = {s: (types_te == s) for s in SUBTASKS}
print("Train:", X_tr.shape, " Test:", X_te.shape)
print("Train class balance:", np.bincount(y_tr, minlength=4))

## 3. Architecture (single 4-class GRU)

In [ ]:
class UnifiedGRU(nn.Module):
    def __init__(self, n_channels=4, hidden_size=8, n_classes=4):
        super().__init__()
        self.gru = nn.GRU(input_size=n_channels, hidden_size=hidden_size, batch_first=True)
        self.readout = nn.Linear(hidden_size, n_classes)
    def forward(self, x):
        x = x.transpose(1, 2)          # (B, channels, time) -> (B, time, channels)
        h, _ = self.gru(x)
        return self.readout(h)         # (B, time, n_classes)

## 4. Training function (returns overall + per-subtask final test accuracy)

In [ ]:
def train_one(hidden_size, seed, n_epochs, lr, batch_size, device):
    torch.manual_seed(seed); np.random.seed(seed)
    loader = DataLoader(TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr)),
                        batch_size=batch_size, shuffle=True)
    X_test_t = torch.from_numpy(X_te).to(device)
    model = UnifiedGRU(4, hidden_size, 4).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(n_epochs):
        model.train()
        for Xb, yb in loader:
            Xb, yb = Xb.to(device), yb.to(device)
            logits = model(Xb); B, T, C = logits.shape
            loss = loss_fn(logits.reshape(B*T, C), yb.unsqueeze(1).expand(B, T).reshape(B*T))
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        pred = model(X_test_t)[:, -1, :].argmax(-1).cpu().numpy()
    overall = float((pred == y_te).mean())
    per_sub = {s: float((pred[TEST_MASKS[s]] == y_te[TEST_MASKS[s]]).mean()) for s in SUBTASKS}
    return overall, per_sub

## 5. Sweep configuration

In [ ]:
HIDDEN_SIZES = list(range(1, 9))   # 1..8
SEEDS = [0, 1, 2, 3, 4]            # 5 seeds for variance
N_EPOCHS = 50
BATCH_SIZE = 64
LR = 1e-3
print("Networks to train:", len(HIDDEN_SIZES)*len(SEEDS), "(", len(HIDDEN_SIZES), "sizes x", len(SEEDS), "seeds )")

## 6. Run the sweep

In [ ]:
sweep = {}        # h -> {"overall": [..seeds..], "per_sub": {subtask: [..seeds..]}}
t0 = time.time()
for h in HIDDEN_SIZES:
    overalls = []; per = {s: [] for s in SUBTASKS}
    ts = time.time()
    for seed in SEEDS:
        ov, ps = train_one(h, seed, N_EPOCHS, LR, BATCH_SIZE, device)
        overalls.append(ov)
        for s in SUBTASKS: per[s].append(ps[s])
    sweep[h] = {"overall": overalls, "per_sub": per}
    print("h=%d  overall mean=%.3f std=%.3f  (%.1fs)" %
          (h, np.mean(overalls), np.std(overalls), time.time()-ts))
print("Total sweep time: %.1f min" % ((time.time()-t0)/60))

## 7. Variance graph (overall accuracy vs hidden units) — use this to pick the hidden size

In [ ]:
means = [np.mean(sweep[h]["overall"]) for h in HIDDEN_SIZES]
stds  = [np.std(sweep[h]["overall"])  for h in HIDDEN_SIZES]
def grp_mean_std(group):
    mu = np.array([np.mean([np.mean(sweep[h]["per_sub"][s]) for s in group]) for h in HIDDEN_SIZES])
    sd = np.array([np.mean([np.std(sweep[h]["per_sub"][s])  for s in group]) for h in HIDDEN_SIZES])
    return mu, sd
det_mu, det_sd = grp_mean_std(DET_SUBTASKS)
loc_mu, loc_sd = grp_mean_std(LOC_SUBTASKS)

fig, ax = plt.subplots(figsize=(9, 6))
ax.errorbar(HIDDEN_SIZES, means, yerr=stds, marker="o", lw=2.5, capsize=5, color="black", label="overall (mean +/- std)")
ax.plot(HIDDEN_SIZES, det_mu, marker="s", ls="--", color="tab:orange", label="detection group mean")
ax.fill_between(HIDDEN_SIZES, det_mu - det_sd, det_mu + det_sd, color="tab:orange", alpha=0.12)
ax.plot(HIDDEN_SIZES, loc_mu, marker="^", ls="--", color="tab:blue", label="localisation group mean")
ax.fill_between(HIDDEN_SIZES, loc_mu - loc_sd, loc_mu + loc_sd, color="tab:blue", alpha=0.12)
ax.axhline(0.25, color="gray", ls=":", alpha=0.5, label="4-class chance")
ax.set_xlabel("Number of hidden units"); ax.set_ylabel("Final test accuracy")
ax.set_xticks(HIDDEN_SIZES); ax.set_ylim(0, 1.05)
ax.set_title("Hidden-unit sweep (5 seeds): variance graph")
ax.grid(alpha=0.3); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

print(f"\n{'Hidden':>8}  {'Mean':>7}  {'Std':>7}  {'Det mean':>10}  {'Loc mean':>10}")
print("-" * 50)
for h, mm, s, d, l in zip(HIDDEN_SIZES, means, stds, det_mu, loc_mu):
    print(f"{h:>8}  {mm:>7.4f}  {s:>7.4f}  {d:>10.4f}  {l:>10.4f}")

## 8. Per-subtask accuracy vs hidden units (two colour groups)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
for s in SUBTASKS:
    m  = np.array([np.mean(sweep[h]["per_sub"][s]) for h in HIDDEN_SIZES])
    sd = np.array([np.std(sweep[h]["per_sub"][s])  for h in HIDDEN_SIZES])
    st = SUBTASK_STYLE[s]
    ax.plot(HIDDEN_SIZES, m, color=st["color"], ls=st["ls"], lw=1.8, label=s)
    ax.fill_between(HIDDEN_SIZES, m - sd, m + sd, color=st["color"], alpha=0.12)
ax.axhline(0.25, color="gray", ls=":", alpha=0.5)
ax.set_xlabel("Number of hidden units"); ax.set_ylabel("Final test accuracy")
ax.set_xticks(HIDDEN_SIZES); ax.set_ylim(0, 1.05)
ax.set_title("Per-subtask accuracy vs hidden units (mean +/- std, 5 seeds; orange=detection, blue=localisation; dashed=conflict)")
ax.grid(alpha=0.3)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

## 9. Numeric summary and recommended hidden size

In [ ]:
print("%3s | %-22s | %-22s | %-22s" % ("h","overall","detection grp","localisation grp"))
print("-"*78)
for h in HIDDEN_SIZES:
    o = sweep[h]["overall"]
    dgrp = [np.mean(sweep[h]["per_sub"][s]) for s in DET_SUBTASKS]
    lgrp = [np.mean(sweep[h]["per_sub"][s]) for s in LOC_SUBTASKS]
    print("%3d | %.3f +/- %.3f        | %.3f                 | %.3f" %
          (h, np.mean(o), np.std(o), np.mean(dgrp), np.mean(lgrp)))

# Recommend the smallest h whose mean overall is within 1% of the best (favouring small, interpretable nets).
best = max(means)
rec = next(h for h, mu in zip(HIDDEN_SIZES, means) if mu >= best - 0.01)
print("\nBest overall mean:", round(best,3), "  Recommended HIDDEN_SIZE =", rec,
      "(smallest size within 1% of best)")
print("Set HIDDEN_SIZE =", rec, "in 02_curriculum_experiments_v5.ipynb")

## 10. Save sweep results

In [ ]:
import pickle
with open(MODEL_DIR/"sweep_results_v8.pkl", "wb") as f:
    pickle.dump(sweep, f)
print("Saved to", MODEL_DIR/"sweep_results_v8.pkl")